# Baseline Machine-Learning Models

This notebook establishes predictive benchmarks for future-return regression and direction classification. It does not contain LSTM models, trading signals, or backtesting.

## Research Rules

Models fit on the training partition. Validation metrics are used for comparison and candidate selection. The test partition remains a final holdout and is evaluated only after a candidate is chosen.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()
if not (project_root / 'ml').exists():
    project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from ml.comparison import compare_classification_models, compare_regression_models
from ml.data.ingestion import MarketDataIngestionService
from ml.data.yahoo import YahooFinanceProvider
from ml.evaluation.classification import evaluate_classification
from ml.evaluation.regression import evaluate_regression
from ml.models.baselines import NaiveDirection, NaiveRegression
from ml.models.boosting import GradientBoostingClassificationModel, GradientBoostingRegressionModel
from ml.models.classification import LogisticDirectionModel
from ml.models.ensemble import RandomForestClassificationModel, RandomForestRegressionModel
from ml.models.regression import LinearRegressionModel
from ml.supervised import build_supervised_dataset

## Load Data and Construct Supervised Datasets

In [ ]:
symbol = 'AAPL'
raw_path = project_root / 'data' / 'raw' / f'{symbol}.csv'
if raw_path.exists():
    ohlcv = pd.read_csv(raw_path, parse_dates=['date'])
else:
    service = MarketDataIngestionService(YahooFinanceProvider())
    ohlcv = service.ingest(symbol, '2020-01-01', '2026-01-01')
regression_dataset = build_supervised_dataset(ohlcv, target_type='regression', horizon=1)
classification_dataset = build_supervised_dataset(ohlcv, target_type='direction', horizon=1)
print(regression_dataset.metadata)

## Validation Comparison: Regression

In [ ]:
regression_results = compare_regression_models(regression_dataset)
regression_table = pd.DataFrame([
    {'model': result.model_name, **result.metrics}
    for result in regression_results
])
regression_table.sort_values('rmse')

## Validation Comparison: Classification

In [ ]:
classification_results = compare_classification_models(classification_dataset)
classification_table = pd.DataFrame([
    {'model': result.model_name, **{key: value for key, value in result.metrics.items() if key != 'confusion_matrix'}}
    for result in classification_results
])
classification_table.sort_values('f1', ascending=False)

## Final Holdout Evaluation

After reviewing validation results, replace these placeholders with an explicitly selected candidate. Do not choose a model by test performance.

In [ ]:
selected_regression = GradientBoostingRegressionModel().fit(regression_dataset.X_train, regression_dataset.y_train)
test_regression_metrics = evaluate_regression(
    regression_dataset.y_test,
    selected_regression.predict(regression_dataset.X_test),
)
selected_classification = GradientBoostingClassificationModel().fit(classification_dataset.X_train, classification_dataset.y_train)
test_classification_metrics = evaluate_classification(
    classification_dataset.y_test,
    selected_classification.predict(classification_dataset.X_test),
    selected_classification.predict_proba(classification_dataset.X_test),
)
test_regression_metrics, test_classification_metrics

## Interpretation and Limitations

These are predictive benchmarks for one historical sample, not evidence of profitability. Directional accuracy is not a trading strategy and does not account for costs, slippage, position sizing, or risk. No LSTM exists yet; future neural models must be compared against these baselines using the same chronological discipline.